In [34]:
import numpy as np
from scipy.optimize import minimize
import time

# --- CONSTANTES PHYSIQUES ---
n_qubits = 12       # Petit système -> simulation exacte possible
dim = 2**n_qubits   # 4096 états
p_layers = 12       # Profondeur du circuit (paramètres à optimiser)
k_sat = 8
ratio = 176.54      # Seuil de transition de phase
n_clauses = int(ratio * n_qubits)

print(f"--- PARTIE 1 : Optimisation des angles (Simulation n={n_qubits}) ---")

# --- 1. SOLVEUR PARFAIT & GÉNÉRATEUR D'INSTANCES ---

def generate_random_8sat(n, m, k=8):
    """Génère m clauses aléatoires (indices et signes)."""
    clauses = []
    for _ in range(m):
        vars_idx = np.random.choice(n, k, replace=False) # Indices
        signs = np.random.choice([-1, 1], k)             # Signes (-1 ou +1)
        clauses.append((vars_idx, signs))
    return clauses

def get_cost_diagonal(n, clauses):
    """
    Calcule le spectre complet de l'Hamiltonien classique (H_C).
    Retourne un vecteur de taille 2^n contenant le nombre de clauses violées pour chaque état.
    """
    cost_diag = np.zeros(2**n)
    
    # Brute force intelligent (vectorisé)
    # On génère la matrice de tous les bitstrings (taille 2^n x n)
    # indices de 0 à 2^n-1
    arange = np.arange(2**n)
    # Bitmasking pour extraire les bits
    # bits[i, j] est le j-ème bit de l'état i
    bits = ((arange[:, None] & (1 << np.arange(n)))) > 0
    spins = 1 - 2 * bits.astype(int) # Map 0->1, 1->-1
    
    for vars_idx, signs in clauses:
        # On extrait les spins des variables concernées par la clause
        # Shape: (2^n, k)
        relevant_spins = spins[:, vars_idx]
        
        # Une clause est satisfaite si au moins un spin match le signe
        # Elle est violée si TOUS les spins sont opposés au signe
        # signs shape: (k,) -> broadcasté
        violation_check = (relevant_spins != signs)
        
        # Si violation_check est True partout pour une ligne, la clause est violée
        clause_violee = np.all(violation_check, axis=1)
        
        cost_diag += clause_violee.astype(float)
        
    return cost_diag

print("Recherche d'une instance SATISFIABLE (Scan complet)...")
while True:
    # 1. Générer une instance aléatoire
    clauses = generate_random_8sat(n_qubits, n_clauses, k_sat)
    
    # 2. Scanner TOUTES les solutions (Solveur Parfait)
    H_C_diag = get_cost_diagonal(n_qubits, clauses)
    
    # 3. Vérifier l'énergie minimale
    min_energy = np.min(H_C_diag)
    
    # On compte le nombre de solutions exactes (E=0)
    n_solutions = np.sum(H_C_diag == 0)
    
    if min_energy == 0:
        print(f"-> Instance trouvée ! (Nombre de solutions exactes : {n_solutions})")
        break
    else:
        # Si min_energy > 0, c'est une instance UNSAT (impossible à résoudre)
        # On rejette et on recommence
        print(f"-> Instance UNSAT (Min Energy={min_energy}). Rejetée.")

# --- 2. SIMULATION QUANTIQUE EXACTE (QAOA) ---

# État initial |+> (superposition uniforme)
psi_0 = np.ones(dim, dtype=np.complex128) / np.sqrt(dim)

def qaoa_success_probability(angles):
    """
    Simule le circuit QAOA et retourne la probabilité de mesurer une solution parfaite.
    Angles: concaténation [beta_0...beta_p, gamma_0...gamma_p]
    """
    betas = angles[:p_layers]
    gammas = angles[p_layers:]
    
    state = psi_0.copy()
    
    for i in range(p_layers):
        beta = betas[i]
        gamma = gammas[i]
        
        # -- Phase Separator (H_C) --
        # Opérateur diagonal : exp(-i * gamma * E_x)
        state *= np.exp(-1j * gamma * H_C_diag)
        
        # -- Mixer (H_B) --
        # Rotation X sur tous les qubits : exp(-i * beta * Sum X)
        # Produit tensoriel de matrices 2x2.
        # Astuce : On utilise la propriété que RX agit indépendamment sur chaque qubit.
        # Code vectorisé rapide pour appliquer RX(2*beta) sur le tenseur d'état.
        
        c = np.cos(beta)
        s = -1j * np.sin(beta)
        
        # On reshape le vecteur d'état pour isoler chaque qubit j
        # C'est équivalent à appliquer la porte sur le j-ème fil
        for j in range(n_qubits):
            # Le vecteur est vu comme (2^(n-1-j), 2, 2^j)
            # L'axe 1 est le qubit j
            shape_before = (1 << (n_qubits - 1 - j), 2, 1 << j)
            state_reshaped = state.reshape(shape_before)
            
            # Application de la matrice [[c, s], [s, c]] sur l'axe 1
            # |0> -> c|0> + s|1>
            # |1> -> s|0> + c|1>
            psi_0_component = state_reshaped[:, 0, :]
            psi_1_component = state_reshaped[:, 1, :]
            
            new_0 = c * psi_0_component + s * psi_1_component
            new_1 = s * psi_0_component + c * psi_1_component
            
            state_reshaped[:, 0, :] = new_0
            state_reshaped[:, 1, :] = new_1
            
            # Pas besoin de reshape back explicite car numpy partage la mémoire
            # mais pour la clarté, le 'state' plat est modifié.
            
    # -- MESURE (Overlap avec le "Solveur Parfait") --
    probs = np.abs(state)**2
    
    # On somme les probabilités uniquement sur les indices identifiés comme solutions (E=0)
    # C'est ici qu'on utilise notre connaissance parfaite de la solution.
    p_success = np.sum(probs[H_C_diag == 0])
    
    return -p_success # Minimisation de l'opposé

# --- 3. OPTIMISATION CLASSIQUE ---

print("\nDémarrage de l'optimisation des angles (COBYLA)...")
t0 = time.time()

# Initialisation "Linear Ramp" (Adiabatique)
# Beta décroît (fort mixage au début -> faible à la fin)
# Gamma croît (faible coût au début -> fort à la fin)
dt = 0.7 
beta_init = [(p_layers - j) * dt * 0.05 for j in range(p_layers)]
gamma_init = [(j + 1) * dt * 0.05 for j in range(p_layers)]
x0 = np.array(beta_init + gamma_init)

# Optimisation
res = minimize(qaoa_success_probability, x0, method='COBYLA', options={'maxiter': 300, 'tol': 1e-4})

t_end = time.time()
print(f"Terminé en {t_end - t0:.2f}s")
print(f"Meilleure probabilité de succès (Overlap) : {-res.fun:.5f}")

# Extraction des résultats pour la partie 2
best_betas = res.x[:p_layers]
best_gammas = res.x[p_layers:]

print("\n--- RÉSULTATS À COPIER POUR PARTIE 2 ---")
print(f"best_betas = {list(best_betas)}")
print(f"best_gammas = {list(best_gammas)}")

--- PARTIE 1 : Optimisation des angles (Simulation n=12) ---
Recherche d'une instance SATISFIABLE (Scan complet)...
-> Instance trouvée ! (Nombre de solutions exactes : 2)

Démarrage de l'optimisation des angles (COBYLA)...
Terminé en 2.31s
Meilleure probabilité de succès (Overlap) : 0.00581

--- RÉSULTATS À COPIER POUR PARTIE 2 ---
best_betas = [0.4808817970075368, 1.2702043623404795, 0.4980535990944026, 1.2435101493835532, 1.2316111562811887, 1.1923270702511115, 0.34244647827787444, 1.3358048398631301, 1.064812703582413, 0.20242098168364878, -0.06803105058913649, 0.15879692562792722]
best_gammas = [-0.06054724414591526, 0.9481625925011377, -0.07472429782924017, 0.12552402305911173, 0.3904673729121482, 0.09834744741867106, 0.0027918671618083766, 0.22598576440812143, 0.2360605214914269, 0.4888404952101768, 0.7870966531187731, 1.1768854192527607]


In [35]:
import numpy as np

# Try importing tqdm for the progress bar, otherwise define a silent fallback
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, desc=None):
        return iterable

def get_scaling_exponent_qaoa_ksat(r, gammas, betas, k, verbose=True):
    """
    Computes the scaling exponent of the QAOA success probability on random k-SAT.
    
    Implements Proposition 3 and Equation 85 (Source 1285) from Boulebnane & Montanaro (2022).
    By including all subsets (including singletons) in the index set A, the 
    fixed-point iteration captures the full scaling exponent (including the prefactor).
    
    Parameters:
    r (float): Clauses-to-variables ratio.
    gammas (list[float]): QAOA gamma angles.
    betas (list[float]): QAOA beta angles.
    k (int): k-SAT parameter (must be a power of 2, k=2^q).
    verbose (bool): If True, shows progress.
    
    Returns:
    float: The scaling exponent C (success prob ~ exp(C*n)).
    """
    
    # --- 1. Validation and Setup ---
    q_val = int(np.log2(k))
    if 2**q_val != k:
        raise ValueError("k must be a power of 2 (k=2^q).")
    
    p = len(gammas)
    dim = 2 * p + 1
    size = 1 << dim
    
    # --- 2. Compute Vectors b and c ---
    # We define b_s and c_alpha according to Example 16 / Equation 85.
    # Crucially, we do NOT mask singletons. The sum runs over ALL subsets alpha.
    
    s_indices = np.arange(size)
    s_bits = [(s_indices >> j) & 1 for j in range(dim)]
    
    # Vector b_s = B_{beta, s} / 2
    # The factor 1/2 absorbs the 1/2^n prefactor from the total probability definition.
    term_sign = (-1.0) ** (s_bits[0] ^ s_bits[p])
    prod_cos = np.ones(size, dtype=np.complex128)
    prod_sin = np.ones(size, dtype=np.complex128)
    
    for j in range(p):
        beta_val = betas[j]
        c_val = np.cos(beta_val / 2)
        s_val = 1j * np.sin(beta_val / 2)
        
        # Exponents for cos and sin based on bit comparisons (Eq 25)
        eq = (1 - (s_bits[j] ^ s_bits[j+1])) + (1 - (s_bits[2*p - j] ^ s_bits[2*p - j - 1]))
        neq = (s_bits[j] ^ s_bits[j+1]) + (s_bits[2*p - j] ^ s_bits[2*p - j - 1])
        
        prod_cos *= (c_val ** eq)
        prod_sin *= (s_val ** neq)
        
    b = (term_sign * prod_cos * prod_sin) / 2.0

    # Vector c_alpha (Eq 16 / 52)
    # We compute this for ALL alpha. The fixed point will naturally handle the linear 
    # terms (singletons) that correspond to the "prefactor".
    c = r * ((-1.0) ** s_bits[p]).astype(np.complex128)
    
    for j in range(dim):
        in_alpha = s_bits[j]
        term = 1.0
        if j < p:
            term = np.exp(-1j * gammas[j] / 2) - 1
        elif j > p:
            gamma_idx = 2 * p - j
            term = np.exp(1j * gammas[gamma_idx] / 2) - 1
        
        mask = (in_alpha == 1)
        c[mask] *= term

    # Note: We do NOT zero out singletons.
    # Precompute (-c)^(1/k)
    neg_c_pow = (-c) ** (1.0 / k)

    # --- 3. Efficient Summation Algorithms ---
    # These compute the matrix-vector products A*v and A^T*v.
    # Since A_{alpha, s} = 1/2 * 1[s compatible with alpha], we must divide by 2.
    
    def algorithm_3_sum_alpha(v):
        """ Computes X_s = sum_{alpha} A_{alpha, s} v_{alpha}. """
        n = dim
        z1 = v.copy()
        # Subset Sum
        for i in range(n):
            shape = (1 << (n - 1 - i), 2, 1 << i)
            z1r = z1.reshape(shape)
            z1r[:, 1, :] += z1r[:, 0, :]
            
        # Superset Sum (via reverse)
        indices = np.arange(size)
        rev_indices = indices ^ (size - 1)
        z0 = v[rev_indices].copy() 
        for i in range(n):
            shape = (1 << (n - 1 - i), 2, 1 << i)
            z0r = z0.reshape(shape)
            z0r[:, 0, :] += z0r[:, 1, :]
        z0 = z0[rev_indices]
        
        # Normalization by 1/2 for A matrix
        return (z0 + z1 - v[0]) / 2.0

    def algorithm_4_sum_s(v):
        """ Computes Y_alpha = sum_{s} A_{alpha, s} v_{s}. """
        # Symmetric to Algorithm 3
        n = dim
        indices = np.arange(size)
        rev_indices = indices ^ (size - 1)
        
        z0 = v[rev_indices].copy()
        z1 = v.copy()
        
        for i in range(n):
            shape = (1 << (n - 1 - i), 2, 1 << i)
            z0r = z0.reshape(shape)
            z0r[:, 0, :] += z0r[:, 1, :]
            z1r = z1.reshape(shape)
            z1r[:, 1, :] += z1r[:, 0, :]
            
        return (z0 + z1 - v[0]) / 2.0

    # --- 4. Fixed Point Iteration (Proposition 3) ---
    # We solve for z* such that z_alpha = -2^q * (dF/dz_alpha)^(k-1)
    
    z_star = np.zeros(size, dtype=np.complex128)
    max_iter = 1000
    tol = 1e-9
    damping = 0.5 
    
    iterator = range(max_iter)
    if verbose:
        iterator = tqdm(iterator, desc="Fixed-point iteration")

    for iteration in iterator:
        # 1. Inner sum X = sum_alpha A * (-c)^1/k * z
        V = neg_c_pow * z_star
        X = algorithm_3_sum_alpha(V)
        
        # 2. Stability Shift
        X_max = np.max(X.real)
        E_stable = np.exp(X - X_max)
        
        # 3. Denominator D
        weighted_E = b * E_stable
        D_stable = np.sum(weighted_E)
        
        if np.abs(D_stable) < 1e-15:
            if verbose: print("Denominator -> 0.")
            break

        # 4. Numerator Y (Gradient * D)
        Y_stable = algorithm_4_sum_s(weighted_E)
        
        # 5. Gradient dF/dz = (-c)^1/k * (Y / D)
        # Shift cancels out
        grad_F = neg_c_pow * (Y_stable / D_stable)
        
        # 6. Update Map (Eq 81): z_new = -2^q * (grad_F)^(k-1)
        # Note the negative sign which comes from the saddle point derivation
        z_new = -k * (grad_F ** (k - 1))
        
        diff = np.max(np.abs(z_star - z_new))
        if diff < tol:
            if verbose and hasattr(iterator, 'close'): iterator.close()
            break
        
        z_star = damping * z_star + (1 - damping) * z_new

    # --- 5. Final Calculation (Eq 13) ---
    # Exponent = F(z*) + (k-1) * sum( (dF/dz)^k )
    
    V = neg_c_pow * z_star
    X = algorithm_3_sum_alpha(V)
    X_max = np.max(X.real)
    E_stable = np.exp(X - X_max)
    
    weighted_E = b * E_stable
    D_stable = np.sum(weighted_E)
    Y_stable = algorithm_4_sum_s(weighted_E)
    
    # F(z*)
    F_val = np.log(D_stable) + X_max
    
    # Gradient term
    grad_F = neg_c_pow * (Y_stable / D_stable)
    
    # Sum term: sum_alpha (dF_alpha)^k
    sum_grad_pow = np.sum(grad_F ** k)
    
    exponent = F_val + (k - 1) * sum_grad_pow
    
    return np.real(exponent)

In [36]:
k = 8 
betas = best_betas
gammas = best_gammas
r = 176.54  
get_scaling_exponent_qaoa_ksat(r, gammas, betas, k)

Fixed-point iteration:   8%|▊         | 75/1000 [19:35<4:01:37, 15.67s/it]


5.030535795289101